<a href="https://colab.research.google.com/github/lp20590-star/Global-Job-Category-Imputation-Labor-Market-Intelligence/blob/main/Global_Job_Category_Imputation_Labor_Market_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Global-Job-Category-Imputation-Labor-Market-Intelligence

Find out what these Unknown jobs actually are

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found += glob.glob("/content/drive/MyDrive/" + pattern + "sample_rows__3__629d0f3b (2).csv")

if len(found) == 0:
    print("❌ Can't find your jobs CSV in Google Drive.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    jobs = pd.read_csv(found[0])
    print(f"✅ Loaded {len(jobs)} jobs from:", FOLDER)


Mounted at /content/drive
✅ Loaded 20 jobs from: /content/drive/MyDrive/MIST 4600/


In [ ]:
def unknown_by_country(jobs):
    unknown = jobs[jobs["category_tag"] == "unknown"]
    return unknown.groupby("country").size()

print(unknown_by_country(jobs))



country
Germany    1
Spain      3
dtype: int64


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import pandas as pd

# Look for the full 4,000-row dfall_clean.csv first, or fall back to the 20-row sample CSV
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found += glob.glob("/content/drive/MyDrive/" + pattern + "dfall_clean.csv")

if len(found) == 0:
    for pattern in ["", "*/", "*/*/", "*/*/*/"]:
        found += glob.glob("/content/drive/MyDrive/" + pattern + "sample_rows__3__629d0f3b*.csv")

if len(found) == 0:
    print("Can't find your jobs CSV in Google Drive.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    jobs = pd.read_csv(found[0])
    print(f"Loaded {len(jobs)} jobs from:", found[0])

In [ ]:
def unknown_by_country(jobs):
    unknown = jobs[jobs["category_tag"] == "unknown"]
    return unknown.groupby("country").size()

print(unknown_by_country(jobs))

In [ ]:
import time
import requests
from google import genai
from google.colab import userdata

ALLOWED = [
    "healthcare & nursing",
    "engineering",
    "trade & construction",
    "it & software development",
    "sales & business development",
    "accounting, banking & finance",
    "logistics, driver & supply chain",
    "teaching & education",
    "hospitality & catering",
    "administrative & office support",
    "hr & recruitment",
    "marketing, pr & advertising",
    "customer service & support",
    "social work & care",
    "maintenance, cleaning & facility",
    "other / specialized",
]

def is_missing_category(tag):
    clean = str(tag).strip().lower()
    if clean in ["unknown", "other-general-jobs", "", "nan"]:
        return True
    return False

class JobPostingAnalyzer:
    def __init__(self, df):
        self.df = df
        self.countries = [
            "Germany", "Switzerland", "France", "Spain",
            "Netherlands", "Italy", "United Kingdom", "USA",
        ]

    def mark_missing(self):
        self.df["is_missing"] = self.df["category_tag"].apply(is_missing_category)
        return self.df

    def missing_rate(self, country):
        sub = self.df[self.df["country"] == country]
        if len(sub) == 0:
            return 0.0
        missing = sub[sub["is_missing"] == True]
        return round((len(missing) / len(sub)) * 100, 1)

# Answer Sub-Question 1
analyzer = JobPostingAnalyzer(jobs)
jobs = analyzer.mark_missing()

for country in analyzer.countries:
    pct = analyzer.missing_rate(country)
    print(f"{country}: {pct}% missing")

# Average salary by country (Week 10 groupby)
print("\nMean salary by country:")
print(jobs.groupby("country")["salary_mid"].mean())

In [ ]:
# Make sure GOOGLE_API_KEY is added in Colab Secrets (key icon on left) with Notebook access ON
key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)

def get_job_category(title, client):
    prompt = f"Pick ONE category from {ALLOWED}. Reply in lowercase only. Job: {title}"
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    answer = "other / specialized"
    if response.text:
        answer = response.text.strip().lower()
    time.sleep(1)

    if answer in ALLOWED:
        return answer
    return "other / specialized"

def classify_row(title):
    return get_job_category(title, client)

# Run on a 10-row sample so it finishes fast in class
sample_size = 10
if len(jobs) < 10:
    sample_size = len(jobs)
sample_jobs = jobs.sample(sample_size).copy()
sample_jobs["imputed_category"] = sample_jobs["title"].apply(classify_row)

# Spot-check 5 rows
show_n = 5
if len(sample_jobs) < 5:
    show_n = len(sample_jobs)
print(sample_jobs[["country", "title", "category_label", "imputed_category"]].sample(show_n))

In [ ]:
import time
from google import genai
from google.colab import userdata

# Add GOOGLE_API_KEY in Colab Secrets (key icon on left) and turn Notebook access ON
key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)

def get_job_category(title, client):
    prompt = f"Pick ONE category from {ALLOWED}. Reply in lowercase only. Job: {title}"
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt,
    )
    answer = "other / specialized"
    if response.text:
        answer = response.text.strip().lower()
    time.sleep(4)

    if answer in ALLOWED:
        return answer
    return "other / specialized"

def classify_row(title):
    return get_job_category(title, client)

# Run on a 5-row sample with 4-second pause (Week 12 Cheat Sheet) to stay under rate limits
sample_size = 5
if len(jobs) < 5:
    sample_size = len(jobs)
sample_jobs = jobs.sample(sample_size).copy()
sample_jobs["imputed_category"] = sample_jobs["title"].apply(classify_row)

# Show all classified sample rows
print(sample_jobs[["country", "title", "category_label", "imputed_category"]])

In [ ]:
import time
import requests

# Compare Gemini vs existing labels on non-missing rows
labeled = sample_jobs[sample_jobs["is_missing"] == False]
orig_list = []
for val in labeled["category_label"]:
    orig_list.append(str(val).strip().lower())

matches = 0
for i, new_cat in enumerate(labeled["imputed_category"]):
    first_word = new_cat.split(" ")[0]
    if first_word in orig_list[i]:
        matches += 1

if len(labeled) > 0:
    print(f"Match rate: {matches}/{len(labeled)}")

# Week 11 Web API: Live EUR exchange rates
url = "https://api.frankfurter.app/latest"
params = {"from": "EUR", "to": "USD,GBP,CHF"}
resp = requests.get(url, params=params)
if resp.status_code == 200:
    print("Live EUR rates:", resp.json()["rates"])

# Week 12 "\n".join() summary for missing German jobs
de_jobs = jobs[jobs["country"] == "Germany"]
de_missing = de_jobs[de_jobs["is_missing"] == True]
if len(de_missing) > 0:
    time.sleep(4)
    titles_text = "\n".join(de_missing.head(10)["title"].astype(str))
    summary = client.models.generate_content(
        model="gemini-3.8-flash",
        contents="In 3 bullets, what jobs are these and what should a job board do?\n" + titles_text,
    )
    print("\nBusiness Summary:\n", summary.text)

# Save output CSV
sample_jobs.to_csv("classified_jobs.csv", index=False)
print("\nSaved classified_jobs.csv")